# Phase 2: Optimizers on a high-dimensional task

Phase 1 used 2-D surfaces so we could *see* the optimizer paths. But the real differences between optimizers show up in higher dimensions, where intuition from a picture runs out. Here we fit a genuinely hard target:

- **Inputs:** 10000 points on the unit sphere in **50 dimensions**.
- **Labels:** produced by a depth-4 **oblique decision tree (ODT)** - 16 leaves carved out by 15 oriented hyperplanes (see [`workshoplib/odt.py`](../workshoplib/odt.py)).
- **Model:** a small fully connected ReLU network trained with **cross-entropy** loss.

We train the same network with five optimizers - SGD, momentum, AdaGrad, Adam, and **MuON** - and compare their loss and accuracy curves. MuON (momentum orthogonalized by a Newton-Schulz iteration) finally has real weight *matrices* to act on, which is why we held it back from Phase 1.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    os.chdir(ROOT.parent)
    ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Working directory:", ROOT)

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
import torch
import matplotlib.pyplot as plt

from workshoplib.datagen import make_odt_classification_data
from workshoplib.model import make_mlp
from workshoplib.optimization import make_optimizer
from workshoplib.training import train_model
from workshoplib import viz

OPTIMIZERS = ["sgd", "momentum", "adagrad", "adam", "muon"]

## 1. Generate the ODT dataset

We draw 10000 labelled points and split them 80/20 into training and validation sets. Train and validation share the *same* decision tree, so validation accuracy measures how well the network generalizes the same target function to unseen points.

In [ ]:
x_train, y_train, x_val, y_val, meta = make_odt_classification_data(
    num_data=10000, dim=50, depth=4, seed=0, val_fraction=0.2
)

print("meta:", meta)
print("train:", tuple(x_train.shape), "val:", tuple(x_val.shape))
print("class balance (train):", round(float(y_train.float().mean()), 3))

A 2-D PCA projection helps build intuition for *why* this is hard. The two classes are thoroughly mixed when squashed down to two dimensions - the ODT boundary lives in the full 50-D space and is not something a straight line (or a tiny network) can capture easily.

In [ ]:
sample = x_train[:2000].numpy()
labels = y_train[:2000].numpy()
centered = sample - sample.mean(axis=0)
_, _, vt = np.linalg.svd(centered, full_matrices=False)
projection = centered @ vt[:2].T

plt.figure(figsize=(6, 5))
plt.scatter(projection[:, 0], projection[:, 1], c=labels, cmap="coolwarm", s=8, alpha=0.5)
plt.title("ODT data, top-2 PCA directions")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.show()

## 2. The model

A small fully connected ReLU classifier: `50 -> 16 -> 16 -> 2`. Two hidden layers of 16 units is deliberately modest - enough to make progress on the ODT target, small enough to run quickly and to make the optimizer differences matter.

In [ ]:
print(make_mlp(input_dim=50, hidden_sizes=(16, 16), num_classes=2))

## 3. Train with each optimizer

We train the identical network (same random initialization, via a fixed seed) with each optimizer, using mini-batch updates. Each optimizer uses a sensible default learning rate from [`workshoplib/optimization.py`](../workshoplib/optimization.py); you can override them later. We record train/validation loss and accuracy every epoch.

In [ ]:
histories = {}
for name in OPTIMIZERS:
    torch.manual_seed(0)  # identical initialization for a fair comparison
    model = make_mlp(input_dim=x_train.shape[1], hidden_sizes=(16, 16), num_classes=2)
    optimizer = make_optimizer(name, model.parameters())
    histories[name] = train_model(
        model, optimizer, x_train, y_train, x_val, y_val, epochs=50, batch_size=128
    )
    print(f"{name:9s} done - final val accuracy {histories[name]['val_acc'][-1]:.3f}")

## 4. Compare the optimizers

Solid lines are training metrics, dashed lines of the same color are validation.

In [ ]:
fig = viz.plot_training_curves(histories)
plt.show()

In [ ]:
print(f"{'optimizer':10s} {'train_acc':>10s} {'val_acc':>10s}")
for name, h in histories.items():
    print(f"{name:10s} {h['train_acc'][-1]:10.3f} {h['val_acc'][-1]:10.3f}")

**What to notice:**

- Plain **SGD** drives the loss down the slowest and ends with the lowest accuracy - a single global step size struggles with the many differently-scaled directions of a 50-D problem.
- **Momentum** and the adaptive methods (**AdaGrad**, **Adam**) fit the training set noticeably faster.
- **MuON** typically reaches the best *validation* accuracy here: orthogonalizing each weight matrix's update spreads learning evenly across all directions of the layer rather than letting a few dominate, which on this task tends to generalize better.
- None of them get close to perfect accuracy - a depth-4 ODT in 50-D is genuinely hard for a small network, which is exactly the point: in high dimensions the choice of optimizer changes both how fast you learn and where you end up.

(Absolute numbers depend on the seed, learning rates, and training length - treat the *ordering and shape* of the curves as the lesson, not the exact values.)

## Try it yourself

- Override a learning rate: `make_optimizer("sgd", model.parameters(), lr=0.5)`.
- Make the network bigger or deeper: `hidden_sizes=(20, 20, 20)` - does more capacity help validation accuracy, or just training accuracy?
- Make the task easier or harder: lower the ODT `depth` (3) or `dim` (e.g. 20) in `make_odt_classification_data`.
- Change the batch size and watch how the curves get noisier (small batches) or smoother (large batches).
- Train longer (`epochs=100`) and see whether the ordering of the optimizers holds up.